# NNUE Training Pipeline

GPU training on Colab. Data is built **locally** (`./dataset.sh` → sharded, pushed to
Drive via rclone); this notebook only **syncs shards → trains → exports `.nbai`**.
No generation, no Stockfish labeling, no browser uploads here.

**Runtime:** GPU (T4 or better). Runtime → Change runtime type → T4 GPU.

**Persistence:** Datasets and checkpoints live on Google Drive, so training resumes
after a session timeout. The repo is cloned to ephemeral `/content` for speed.

---
## ⚙️ 1 — Setup

In [ ]:
# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Configure these paths once ───────────────────────────────────────────────
REPO_URL   = 'https://git.janis-eccarius.de/NowChess/NowChessSystems.git'
DRIVE_ROOT = '/content/drive/MyDrive/NowChess'   # datasets + weights persist here
REPO_DIR   = '/content/NowChessSystems'          # ephemeral, fast local clone
PYTHON_DIR = f'{REPO_DIR}/modules/official-bots/python'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_ROOT, exist_ok=True)

# Clone to ephemeral /content (NOT Drive) — fast checkout, no Drive bloat.
if not os.path.isdir(REPO_DIR):
    !git clone --depth=1 "{REPO_URL}" "{REPO_DIR}"
    print('Repo cloned to /content.')
else:
    !git -C "{REPO_DIR}" pull --ff-only
    print('Repo updated.')

In [ ]:
# Install Python dependencies. No Stockfish — labeling happens on the local box,
# this notebook only trains on already-labeled shards.
!pip install -q chess tqdm rich zstandard

import sys
sys.path.insert(0, f'{PYTHON_DIR}/src')
sys.path.insert(0, PYTHON_DIR)
print('Python path configured.')

---
## 🗄️ 2 — Data

Datasets are built **locally** (`./dataset.sh`) and pushed to Drive with rclone as
compressed shards under `MyDrive/NowChess/datasets/`. Here we just sync those shards
to the fast local disk — no generation, no labeling, no browser uploads.

The cell reads `manifest.json` and copies only shards not already cached on `/content`.

In [ ]:
import json, shutil
from pathlib import Path

# Source: shards synced from the local box via `rclone copy datasets/ gdrive:NowChess/datasets`
DRIVE_DATASETS = Path(DRIVE_ROOT) / 'datasets'
LOCAL_DATASETS = Path('/content/datasets')
(LOCAL_DATASETS / 'shards').mkdir(parents=True, exist_ok=True)

manifest = json.load(open(DRIVE_DATASETS / 'manifest.json'))
print(f"Dataset v{manifest['dataset_version']}: "
      f"{manifest['total_positions']:,} positions across {len(manifest['shards'])} shards")

copied = 0
for sh in manifest['shards']:
    dst = LOCAL_DATASETS / 'shards' / sh['file']
    if not dst.exists():                      # cache: only copy shards we don't already have
        shutil.copy(DRIVE_DATASETS / 'shards' / sh['file'], dst)
        copied += 1
shutil.copy(DRIVE_DATASETS / 'manifest.json', LOCAL_DATASETS / 'manifest.json')

DATA_PATH = str(LOCAL_DATASETS)               # train_nnue / burst_train read this dir of shards directly
print(f"Synced {copied} new shard(s). Dataset ready at {DATA_PATH}")

---
## 🏋️ 3 — Train

Standard training runs a fixed number of epochs.  
**Burst mode** is better for Colab: it repeatedly restarts from the best checkpoint within a time budget, surviving session disconnects gracefully.

In [ ]:
from train import train_nnue, burst_train, DEFAULT_HIDDEN_SIZES

WEIGHTS_DIR = Path(DRIVE_ROOT) / 'weights'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = str(WEIGHTS_DIR / 'nnue_weights.pt')

# ── Training hyperparameters ──────────────────────────────────────────────────
HIDDEN_SIZES      = DEFAULT_HIDDEN_SIZES
# fen_to_features builds a DENSE 98304-dim input, so a batch costs
# batch_size * 98304 * 4 bytes on the host (× DataLoader prefetch). On Colab's
# ~12 GB RAM keep this small; raise it only if you have headroom.
BATCH_SIZE        = 4096
EPOCHS            = 100
EARLY_STOPPING    = 10                     # None to disable
SUBSAMPLE_RATIO   = 1.0

# Resume from latest checkpoint if one exists
checkpoints = sorted(WEIGHTS_DIR.glob('nnue_weights_v*.pt'))
CHECKPOINT = str(checkpoints[-1]) if checkpoints else None
if CHECKPOINT:
    print(f'Resuming from checkpoint: {CHECKPOINT}')
else:
    print('Starting training from scratch.')

In [ ]:
# ── Standard training ─────────────────────────────────────────────────────────
# Use this when you have a reliable long-running session.

train_nnue(
    data_file=DATA_PATH,
    output_file=OUTPUT_FILE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    checkpoint=CHECKPOINT,
    use_versioning=True,
    early_stopping_patience=EARLY_STOPPING,
    subsample_ratio=SUBSAMPLE_RATIO,
    hidden_sizes=HIDDEN_SIZES,
)

In [ ]:
# ── Burst training (recommended for Colab free tier) ─────────────────────────
# Restarts from the global best each time early stopping fires.
# Set BURST_MINUTES to slightly less than the Colab session limit (~70 min).

BURST_MINUTES      = 70
EPOCHS_PER_SEASON  = 30
BURST_PATIENCE     = 8

burst_train(
    data_file=DATA_PATH,
    output_file=OUTPUT_FILE,
    duration_minutes=BURST_MINUTES,
    epochs_per_season=EPOCHS_PER_SEASON,
    early_stopping_patience=BURST_PATIENCE,
    batch_size=BATCH_SIZE,
    initial_checkpoint=CHECKPOINT,
    use_versioning=True,
    subsample_ratio=SUBSAMPLE_RATIO,
    hidden_sizes=HIDDEN_SIZES,
)

---
## 📦 4 — Export

Convert the best `.pt` checkpoint to the `.nbai` binary format read by `NbaiLoader` in Scala.

In [ ]:
from export import export_to_nbai

NBAI_FILE = Path(DRIVE_ROOT) / 'nnue_weights.nbai'

# Pick the latest versioned checkpoint
checkpoints = sorted(WEIGHTS_DIR.glob('nnue_weights_v*.pt'))
if not checkpoints:
    raise FileNotFoundError('No checkpoints found in ' + str(WEIGHTS_DIR))

latest = checkpoints[-1]
print(f'Exporting {latest.name} → {NBAI_FILE.name}')

export_to_nbai(
    weights_file=str(latest),
    output_file=str(NBAI_FILE),
    trained_by='colab',
)
print('Export complete.')

---
## ⬇️ 5 — Download

Download the `.nbai` weights file and the latest `.pt` checkpoint to your local machine.

Place `nnue_weights.nbai` in `modules/official-bots/src/main/resources/` and rebuild the native image.

In [ ]:
from google.colab import files

if NBAI_FILE.exists():
    files.download(str(NBAI_FILE))
    print(f'Downloading {NBAI_FILE.name}')
else:
    print('No .nbai file found — run the Export cell first.')

checkpoints = sorted(WEIGHTS_DIR.glob('nnue_weights_v*.pt'))
if checkpoints:
    latest = checkpoints[-1]
    files.download(str(latest))
    print(f'Downloading checkpoint {latest.name}')
else:
    print('No .pt checkpoint found.')